<a href="https://colab.research.google.com/github/MRharu0310/UVR5_Google-Colab/blob/main/UVR5_Google_Colab_v1.0.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UVR5 / BS-Roformer — Google Colab


処理の基本構成：
```text
元音声 M
    │
    ▼
抽出 Vocal V
    │
    ▼
Residual = M - V
    │
    ▼
Vocal activityを解析
    │
    ├── Vocalが少ない区間
    │       └── Residualの楽器レベルを基準値として測定
    │
    └── Vocalが多い区間
            └── Residualの現在の楽器レベルを測定
                    │
                    ▼
             基準レベルとの差を推定
                    │
                    ▼
               Gain Envelope
                    │
                  平滑化
                    │
                    ▼
              Residual × Gain
                    │
                    ▼
             補正済みInstrumental
```

## 0. 作業マシンの確認 / 6STEM設定

1. **ランタイムのタイプを変更**から **Python3, T4 GPU, 最新Ver.** を選択する

2. 抽出後の音声ファイルを各楽器にさらに分離する時はENABLE_6STEMをTrueにする

3. すべてのセルを実行を押す

4. 「5. 音声のアップロード」でファイルのアップロードができるようになるので、使用するWAVファイルを選択する


In [ ]:
ENABLE_6STEM = False # 各楽器に分離する時はTrueにする

In [ ]:
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Colab, choose Runtime → Change runtime type → GPU and reconnect.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## 1. 依存パッケージのインストール



In [ ]:
from pathlib import Path
import subprocess
import sys

SETUP_MARKER = Path("/content/uvr_vocal_aware/.packages_ready")
SETUP_MARKER.parent.mkdir(parents=True, exist_ok=True)

if SETUP_MARKER.exists():
    print("パッケージセットアップ済みです。再インストールをスキップします。")
else:
    print("初回セットアップを開始します。")

    subprocess.run(
        ["apt-get", "-qq", "update"],
        check=True
    )

    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "ffmpeg"],
        check=True
    )

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "bs-roformer-infer",
            "soundfile",
            "scipy",
            "matplotlib",
            "pyyaml",
            "huggingface_hub",
        ],
        check=True
    )

    SETUP_MARKER.write_text("ready", encoding="utf-8")
    print("パッケージセットアップ完了。")

print("このセルは同一ランタイムでは1回だけ実質的に処理されます。")


## 2. 分岐A: `bs_roformer_revive2.ckpt` のダウンロードと検証

制作者本人(unwa氏, HF上のアカウント名は pcunwa)が直接ホストしている一次配布元から取得し、SHA-256を検証します。

In [ ]:
from pathlib import Path
import hashlib
import urllib.request

MODEL_DIR = Path("/content/bs_roformer_revive2")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CKPT = MODEL_DIR / "bs_roformer_revive2.ckpt"
CONFIG = MODEL_DIR / "config.yaml"

CKPT_URL = "https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/bs_roformer_revive2.ckpt"
CONFIG_URL = "https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/config.yaml"
EXPECTED_SHA256 = "58098850c882a7472dad39f99fb8040ce6eaafe671cfe9881d89aea276bbb5f5"

if CKPT.exists():
    print("Revive V2 checkpoint exists. Download skipped.")
else:
    print("Downloading Revive V2 checkpoint...")
    urllib.request.urlretrieve(CKPT_URL, CKPT)

if CONFIG.exists():
    print("Revive V2 config exists. Download skipped.")
else:
    print("Downloading Revive V2 config...")
    urllib.request.urlretrieve(CONFIG_URL, CONFIG)

h = hashlib.sha256()
with CKPT.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(chunk)

actual_sha256 = h.hexdigest()
print("Checkpoint:", CKPT)
print("Config:", CONFIG)
print("SHA-256:", actual_sha256)

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Checkpoint SHA-256 mismatch. Hugging Face側でファイルが更新された可能性があります。"
    )

print("SHA-256 verification: OK")


## 3. Revive2の元設定を確認

In [ ]:
print(CONFIG.read_text())

## 4. Revive2用の推論設定を構築

- `chunk_size` / `dim_t`: 変更しない(公開されているアーキテクチャ値のまま)
- `num_overlap`: 2 → **4**(音切れ対策)
- `batch_size`: 2(CUDA OOMが出る場合のみ1に下げる)


In [ ]:
import yaml
from bs_roformer.inference import SafeLoaderWithTuple

INFERENCE_BATCH_SIZE = 2
INFERENCE_OVERLAP = 4

with CONFIG.open() as f:
    cfg = yaml.load(f, Loader=SafeLoaderWithTuple)

cfg["inference"]["batch_size"] = INFERENCE_BATCH_SIZE
cfg["inference"]["num_overlap"] = INFERENCE_OVERLAP

RUN_CONFIG = MODEL_DIR / "config_colab_improved.yaml"
with RUN_CONFIG.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("batch_size =", INFERENCE_BATCH_SIZE)
print("num_overlap =", INFERENCE_OVERLAP)
print("runtime config =", RUN_CONFIG)


**2曲目以降**：音声のアップロードから順番に実行します。Cell 4のパッケージインストールやCell 6のモデル取得をやり直す必要はありません。

※ランタイムを再起動した場合は `/content` のモデル・パッケージ環境が失われるため、初回セットアップから再実行してください。


## 5. 音声のアップロード
32-bit float WAV・44.1kHz・ステレオに変換します。


In [ ]:
from google.colab import files
from pathlib import Path
import subprocess
from datetime import datetime
import re

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No audio file was uploaded.")

raw_name = next(iter(uploaded))
raw_path = Path("/content") / raw_name

safe_stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", raw_path.stem).strip("._-") or "audio"
RUN_ROOT = (
    Path("/content/uvr_vocal_aware/runs")
    / f"{datetime.now().strftime('%Y%m%d_%H%M%S_%f')[:-3]}_{safe_stem}"
)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

INPUT_DIR = RUN_ROOT / "input"
OUTPUT_DIR_REVIVE2 = RUN_ROOT / "revive"
OUTPUT_DIR_SW = RUN_ROOT / "sw_raw"
PROCESSED_DIR = RUN_ROOT / "final"

for d in [INPUT_DIR, OUTPUT_DIR_REVIVE2, OUTPUT_DIR_SW, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_WAV = INPUT_DIR / (raw_path.stem + ".wav")

# CPU処理: ffmpegによるサンプルレート/チャンネル/float32統一
subprocess.run([
    "ffmpeg", "-y", "-i", str(raw_path),
    "-ar", "44100", "-ac", "2", "-c:a", "pcm_f32le", str(INPUT_WAV)
], check=True)

print("Run directory:", RUN_ROOT)
print("Input:", INPUT_WAV)
print("このセルから下を順番に実行すると、1ファイル分の処理が完了します。")
print("別ファイルを処理するときは、このセルを再実行してください。")


## 6. BS-Roformer Revive V2 を実行(CUDA)



In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-m", "bs_roformer.inference",
    "--model_type", "bs_roformer",
    "--config_path", str(RUN_CONFIG),
    "--model_path", str(CKPT),
    "--input_folder", str(INPUT_DIR),
    "--store_dir", str(OUTPUT_DIR_REVIVE2),
    "--device", "cuda:0",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 7. Revive2の出力を確認


In [ ]:
raw_wavs = sorted(OUTPUT_DIR_REVIVE2.glob("*.wav"))
for p in raw_wavs:
    print(p.name)

if not raw_wavs:
    raise RuntimeError("No output WAV files were produced by Revive2.")


## 8. Instrumentalの残差候補を作成

`instrumental_residual = 元の音源 − 推定ボーカル`


In [ ]:
import numpy as np
import soundfile as sf
from scipy.signal import correlate

def load_audio(path):
    x, sr = sf.read(path, always_2d=True, dtype="float32")
    return x, sr

mix, sr_mix = load_audio(INPUT_WAV)

vocal_candidates = [p for p in raw_wavs if "vocal" in p.name.lower()]
if not vocal_candidates:
    raise RuntimeError("Could not identify a vocals output. Inspect the raw output filenames above.")

VOCALS_WAV = vocal_candidates[0]
vocals, sr_v = load_audio(VOCALS_WAV)

if sr_mix != sr_v:
    raise RuntimeError(f"Sample-rate mismatch: mix={sr_mix}, vocals={sr_v}")

n = min(len(mix), len(vocals))
mix = mix[:n]
vocals = vocals[:n]


def estimate_lag_samples(a, b, sr, max_shift_sec=0.02, window_sec=10.0):
    # 中央付近の数秒を使って a に対する b のサンプルずれを相互相関で推定する
    # 戻り値が正なら b が a より遅れている
    a_mono = a.mean(axis=1) if a.ndim > 1 else a
    b_mono = b.mean(axis=1) if b.ndim > 1 else b

    center = len(a_mono) // 2
    half_win = int(min(window_sec * sr, len(a_mono) / 2) // 2)
    seg_a = a_mono[center - half_win: center + half_win]
    seg_b = b_mono[center - half_win: center + half_win]

    max_shift = int(max_shift_sec * sr)
    corr = correlate(seg_a, seg_b, mode="full")
    lags = np.arange(-len(seg_b) + 1, len(seg_a))
    valid = (lags >= -max_shift) & (lags <= max_shift)
    best_lag = int(lags[valid][np.argmax(corr[valid])])
    return best_lag


lag = estimate_lag_samples(mix[:, 0], vocals[:, 0], sr_mix)
print(f"推定サンプルずれ(mixに対するvocalsのずれ): {lag} samples "
      f"({lag / sr_mix * 1000:.3f} ms)")

if lag == 0:
    print("ずれは検出されませんでした。位相干渉によるこもり・音量変動の可能性は低いと考えられます。")
    vocals_aligned = vocals
else:
    print("ずれが検出されました。位相干渉(コムフィルタ)による音質劣化を避けるため、補正してから残差を計算します。")
    if lag > 0:
        vocals_aligned = np.vstack([vocals[lag:], np.zeros((lag, vocals.shape[1]), dtype=vocals.dtype)])
    else:
        shift = -lag
        vocals_aligned = np.vstack([np.zeros((shift, vocals.shape[1]), dtype=vocals.dtype), vocals[:-shift]])

residual = mix - vocals_aligned

RESIDUAL_WAV = PROCESSED_DIR / f"{INPUT_WAV.stem}_instrumental_residual.wav"
sf.write(RESIDUAL_WAV, residual, sr_mix, subtype="FLOAT")

print("Vocals:", VOCALS_WAV)
print("Residual instrumental:", RESIDUAL_WAV)


## 9. 音量変動(pumping)の可視化


In [ ]:
import matplotlib.pyplot as plt

def rms_envelope_db(x, sr, window_sec=1.0):
    x_mono = x.mean(axis=1) if x.ndim > 1 else x
    win = max(int(window_sec * sr), 1)
    n_win = len(x_mono) // win
    env = np.array([
        np.sqrt(np.mean(x_mono[i * win:(i + 1) * win] ** 2) + 1e-12)
        for i in range(n_win)
    ])
    env_db = 20 * np.log10(env + 1e-12)
    t = np.arange(n_win) * window_sec
    return t, env_db

direct_instrumentals_preview = [
    p for p in raw_wavs
    if any(k in p.name.lower() for k in ["instrumental", "other"])
]

plt.figure(figsize=(12, 4))

if direct_instrumentals_preview:
    d, sr_d = load_audio(direct_instrumentals_preview[0])
    t_d, env_d = rms_envelope_db(d, sr_d)
    plt.plot(t_d, env_d, label=f"direct model instrumental ({direct_instrumentals_preview[0].name})")

t_r, env_r = rms_envelope_db(residual, sr_mix)
plt.plot(t_r, env_r, label="residual instrumental (mix - vocals, aligned)")

plt.xlabel("time [s]")
plt.ylabel("RMS level [dBFS] (1s window)")
plt.title("Instrumental level over time — periodic dips may indicate chunk-boundary pumping")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 10. Instrumentalのdenoise(任意)

シンバル・ハイハット・ルームアンビエンスなど高域の情報が重要な場合はOFFのままにしてください。

**デフォルトOFF**


In [ ]:
import subprocess

ENABLE_DENOISE = False #任意
DENOISE_NOISE_FLOOR = -28

DENOISED_WAV = PROCESSED_DIR / f"{INPUT_WAV.stem}_instrumental_residual_denoised.wav"

if ENABLE_DENOISE:
    subprocess.run([
        "ffmpeg", "-y", "-i", str(RESIDUAL_WAV),
        "-af", f"afftdn=nf={DENOISE_NOISE_FLOOR}:tn=1",
        "-c:a", "pcm_f32le", str(DENOISED_WAV)
    ], check=True)
    print("Denoised:", DENOISED_WAV)
else:
    print("Denoise is OFF. Set ENABLE_DENOISE = True and rerun this cell to test it.")


## 11. Vocal-aware Level Recovery


In [ ]:
import numpy as np
import soundfile as sf
from scipy.ndimage import gaussian_filter1d

ENABLE_VOCAL_AWARE_RECOVERY = True

# Vocal activity
VOCAL_ACTIVITY_FRAME_SEC = 0.25
VOCAL_ACTIVITY_THRESHOLD_REL_DB = -35.0
VOCAL_ACTIVITY_FULL_REL_DB = -15.0

# Vocalが少ない区間から目標レベルを推定する局所範囲
LOCAL_REFERENCE_SEC = 1.5
MIN_INACTIVE_NEIGHBORS = 3

# 補正量
MAX_VOCAL_RECOVERY_DB = 18.0
GAIN_SMOOTH_SEC = 0.05
MIN_DEFICIT_DB = 0.5

_enable_denoise = bool(globals().get("ENABLE_DENOISE", False))
_denoised_wav = globals().get("DENOISED_WAV", None)

if _enable_denoise and _denoised_wav is not None and _denoised_wav.exists():
    _source_for_recovery = _denoised_wav
    print("Recovery source: DENOISED_WAV")
else:
    _source_for_recovery = RESIDUAL_WAV
    print("Recovery source: RESIDUAL_WAV")

source_audio, source_sr = load_audio(_source_for_recovery)
vocals_audio, vocals_sr = load_audio(VOCALS_WAV)

if source_sr != vocals_sr:
    raise RuntimeError(
        f"Sample-rate mismatch: instrumental={source_sr}, vocals={vocals_sr}"
    )

n = min(len(source_audio), len(vocals_audio))
source_audio = source_audio[:n]
vocals_audio = vocals_audio[:n]


def frame_rms(x, sr, frame_sec):
    """非重複フレームごとのRMSを計算する。"""
    mono = x.mean(axis=1) if x.ndim > 1 else x

    frame_len = max(int(round(frame_sec * sr)), 1)
    n_frames = int(np.ceil(len(mono) / frame_len))

    padded = np.pad(
        mono,
        (0, n_frames * frame_len - len(mono)),
        mode="constant"
    )

    frames = padded.reshape(n_frames, frame_len)

    rms = np.sqrt(
        np.mean(frames ** 2, axis=1) + 1e-12
    )

    centers_sec = (
        np.arange(n_frames) + 0.5
    ) * frame_sec

    return centers_sec, rms

t_frames, residual_rms = frame_rms(
    source_audio,
    source_sr,
    VOCAL_ACTIVITY_FRAME_SEC
)

_, vocal_rms = frame_rms(
    vocals_audio,
    vocals_sr,
    VOCAL_ACTIVITY_FRAME_SEC
)

residual_db = 20.0 * np.log10(residual_rms + 1e-12)
vocal_db = 20.0 * np.log10(vocal_rms + 1e-12)

# Vocal全体の相対基準
vocal_reference_db = float(
    np.percentile(vocal_db, 95)
)

activity_threshold = max(
    -80.0,
    vocal_reference_db + VOCAL_ACTIVITY_THRESHOLD_REL_DB
)

activity_full = max(
    activity_threshold + 1.0,
    vocal_reference_db + VOCAL_ACTIVITY_FULL_REL_DB
)

vocal_activity = np.clip(
    (vocal_db - activity_threshold)
    / max(activity_full - activity_threshold, 1e-6),
    0.0,
    1.0
)

# Vocalが少ない / 多い区間
active_mask = vocal_activity >= 0.75
inactive_mask = vocal_activity <= 0.15

print()
print("========== VOCAL ACTIVITY ==========")
print(f"vocal 95%             : {vocal_reference_db:.2f} dB")
print(f"activity threshold    : {activity_threshold:.2f} dB")
print(f"activity full         : {activity_full:.2f} dB")
print(f"activity min          : {np.min(vocal_activity):.3f}")
print(f"activity max          : {np.max(vocal_activity):.3f}")
print(f"activity mean         : {np.mean(vocal_activity):.3f}")
print(f"active frames         : {active_mask.sum()}")
print(f"inactive frames       : {inactive_mask.sum()}")
print("====================================")
print()

if inactive_mask.sum() < MIN_INACTIVE_NEIGHBORS:
    raise RuntimeError(
        "Vocalが少ない区間が少なすぎるため、"
        "楽器の基準レベルを安全に推定できません。"
    )

reference_radius_frames = max(
    int(round(
        LOCAL_REFERENCE_SEC
        / VOCAL_ACTIVITY_FRAME_SEC
    )),
    1
)

reference_db = np.full(
    len(residual_db),
    np.nan,
    dtype=np.float64
)

global_inactive_reference_db = float(
    np.median(residual_db[inactive_mask])
)

for i in range(len(residual_db)):

    start = max(
        0,
        i - reference_radius_frames
    )

    end = min(
        len(residual_db),
        i + reference_radius_frames + 1
    )

    local_inactive = inactive_mask[start:end]

    if np.count_nonzero(local_inactive) >= MIN_INACTIVE_NEIGHBORS:
        local_values = residual_db[start:end][local_inactive]

        # 外れ値の影響を抑えるため中央値を使用
        reference_db[i] = float(
            np.median(local_values)
        )
    else:
        # 周辺に十分な非Vocal区間がない場合は、
        # 全体の非Vocal区間から求めた基準値を使用
        reference_db[i] = global_inactive_reference_db

current_db = residual_db.copy()

deficit_db = (
    reference_db - current_db
)

deficit_db = np.maximum(
    deficit_db,
    0.0
)

deficit_db[
    deficit_db < MIN_DEFICIT_DB
] = 0.0

gain_db_frames = (
    deficit_db * vocal_activity
)

gain_db_frames = np.clip(
    gain_db_frames,
    0.0,
    MAX_VOCAL_RECOVERY_DB
)

sigma_frames = max(
    GAIN_SMOOTH_SEC
    / VOCAL_ACTIVITY_FRAME_SEC,
    0.5
)

gain_db_frames = gaussian_filter1d(
    gain_db_frames.astype(np.float64),
    sigma=sigma_frames,
    mode="nearest"
)

gain_db_frames = np.clip(
    gain_db_frames,
    0.0,
    MAX_VOCAL_RECOVERY_DB
)

if not ENABLE_VOCAL_AWARE_RECOVERY:
    gain_db_frames[:] = 0.0

print()
print("========== VOCAL-AWARE LEVEL RECOVERY ==========")
print(f"reference radius      : {LOCAL_REFERENCE_SEC:.2f} sec")
print(f"minimum inactive      : {MIN_INACTIVE_NEIGHBORS}")
print(f"global inactive ref  : {global_inactive_reference_db:.2f} dB")
print(f"deficit max           : {np.max(deficit_db):.2f} dB")
print(f"gain min              : {np.min(gain_db_frames):.2f} dB")
print(f"gain max              : {np.max(gain_db_frames):.2f} dB")
print(f"gain mean             : {np.mean(gain_db_frames):.2f} dB")
print("=================================================")
print()

gain_linear_frames = (
    10.0 ** (gain_db_frames / 20.0)
)

sample_times = (
    np.arange(n, dtype=np.float64)
    / source_sr
)

gain_linear_samples = np.interp(
    sample_times,
    t_frames,
    gain_linear_frames,
    left=float(gain_linear_frames[0]),
    right=float(gain_linear_frames[-1])
).astype(np.float32)

recovered = (
    source_audio
    * gain_linear_samples[:, None]
)

RECOVERED_WAV = (
    PROCESSED_DIR
    / f"{INPUT_WAV.stem}_instrumental_vocal_recovered.wav"
)

sf.write(
    RECOVERED_WAV,
    recovered,
    source_sr,
    subtype="FLOAT"
)

print(
    "Vocal-aware recovered instrumental:",
    RECOVERED_WAV
)

## 12. Vocal-aware recovery の結果を確認

In [ ]:
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt

ENABLE_PEAK_GUARD = True
TARGET_PEAK_DBFS = -1.0

SOURCE_FOR_EXPORT = (
    RECOVERED_WAV
    if ENABLE_VOCAL_AWARE_RECOVERY and RECOVERED_WAV.exists()
    else (
        DENOISED_WAV
        if ENABLE_DENOISE and DENOISED_WAV.exists()
        else RESIDUAL_WAV
    )
)

EXPORT_WAV = (
    PROCESSED_DIR
    / f"{INPUT_WAV.stem}_instrumental_final.wav"
)

audio, sr = sf.read(
    SOURCE_FOR_EXPORT,
    always_2d=True,
    dtype="float32"
)

if ENABLE_PEAK_GUARD:
    peak = float(np.max(np.abs(audio)))
    target = 10 ** (TARGET_PEAK_DBFS / 20.0)

    if peak > target and peak > 0:
        audio = audio * (target / peak)

sf.write(
    EXPORT_WAV,
    audio,
    sr,
    subtype="FLOAT"
)

t_before, env_before = rms_envelope_db(
    source_audio,
    source_sr,
    window_sec=0.25
)

t_after, env_after = rms_envelope_db(
    audio,
    sr,
    window_sec=0.25
)

has_vocal_activity = (
    "vocal_activity" in globals()
    and "t_frames" in globals()
)

plt.figure(figsize=(14, 6))

plt.plot(
    t_before,
    env_before,
    label="BEFORE vocal-aware recovery",
    linewidth=2.0,
    alpha=0.65
)

plt.plot(
    t_after,
    env_after,
    label="AFTER vocal-aware recovery",
    linewidth=1.5,
    linestyle="--"
)

if has_vocal_activity:
    vocal_plot = -60.0 + 60.0 * vocal_activity

    plt.plot(
        t_frames,
        vocal_plot,
        label="Vocal activity",
        linewidth=1.0,
        alpha=0.5
    )

plt.xlabel("Time [s]")
plt.ylabel("Level [dBFS]")
plt.title(
    "Instrumental Level: Before vs After Vocal-Aware Recovery"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))

plt.plot(
    t_before,
    env_before,
    label="BEFORE vocal-aware recovery",
    linewidth=1.5
)

plt.xlabel("Time [s]")
plt.ylabel("Level [dBFS]")
plt.title("Instrumental Level BEFORE Vocal-Aware Recovery")

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))

plt.plot(
    t_after,
    env_after,
    label="AFTER vocal-aware recovery",
    linewidth=1.5
)

plt.xlabel("Time [s]")
plt.ylabel("Level [dBFS]")
plt.title("Instrumental Level AFTER Vocal-Aware Recovery")

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

if has_vocal_activity and "gain_db_frames" in globals():

    plt.figure(figsize=(14, 4))

    plt.plot(
        t_frames,
        gain_db_frames,
        label="Applied recovery gain",
        linewidth=1.5
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Gain [dB]")
    plt.title("Vocal-Aware Recovery Gain")

    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


print("Final instrumental:", EXPORT_WAV)

In [ ]:
print("========== VOCAL RECOVERY DEBUG ==========")

print(f"vocal_db min       : {np.min(vocal_db):.2f} dB")
print(f"vocal_db max       : {np.max(vocal_db):.2f} dB")
print(f"vocal_db median    : {np.median(vocal_db):.2f} dB")
print(f"vocal_db 95%       : {np.percentile(vocal_db, 95):.2f} dB")

print()
print(f"activity_threshold : {activity_threshold:.2f} dB")
print(f"activity_full      : {activity_full:.2f} dB")

print()
print(f"vocal_activity min : {np.min(vocal_activity):.4f}")
print(f"vocal_activity max : {np.max(vocal_activity):.4f}")
print(f"vocal_activity mean: {np.mean(vocal_activity):.4f}")

print()
print(f"active frames      : {active_mask.sum()}")
print(f"inactive frames    : {inactive_mask.sum()}")

if "deficit_db" in globals():
    print()
    print(f"deficit min        : {np.min(deficit_db):.4f} dB")
    print(f"deficit max        : {np.max(deficit_db):.4f} dB")
    print(f"deficit mean       : {np.mean(deficit_db):.4f} dB")

if "gain_db_frames" in globals():
    print()
    print(f"gain min           : {np.min(gain_db_frames):.4f} dB")
    print(f"gain max           : {np.max(gain_db_frames):.4f} dB")
    print(f"gain mean          : {np.mean(gain_db_frames):.4f} dB")

print("==========================================")

## 13. BS-Roformer-SW(6-stem)のダウンロード


In [ ]:
# 6-stemを使う場合だけ、このセル以降を実行します。
if ENABLE_6STEM:
    from bs_roformer import MODEL_REGISTRY

    SW_SLUG = "roformer-model-bs-roformer-sw-by-jarredou"
    sw_entry = MODEL_REGISTRY.get(SW_SLUG)

    if sw_entry is None:
        raise RuntimeError(f"6-stem model not found in MODEL_REGISTRY: {SW_SLUG}")

    print("slug:", sw_entry.slug)
    print("checkpoint filename:", sw_entry.checkpoint)
    print("config filename:", sw_entry.config)
else:
    print("ENABLE_6STEM=False: BS-Roformer-SWのセットアップをスキップします。")


In [ ]:
if ENABLE_6STEM:
    import subprocess

    SW_MODELS_ROOT = Path("/content/models")
    SW_MODEL_DIR = SW_MODELS_ROOT / SW_SLUG
    SW_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    SW_CKPT = SW_MODEL_DIR / sw_entry.checkpoint
    SW_CONFIG = SW_MODEL_DIR / sw_entry.config

    if SW_CKPT.exists() and SW_CONFIG.exists():
        print("BS-Roformer-SW model exists. Download skipped.")
    else:
        subprocess.run([
            "bs-roformer-download",
            "--model", SW_SLUG,
            "--output-dir", str(SW_MODELS_ROOT),
        ], check=True)

    if not SW_CKPT.exists() or not SW_CONFIG.exists():
        raise RuntimeError(
            f"想定したパスにファイルが見つかりません: {SW_MODEL_DIR}"
        )

    print("SW checkpoint:", SW_CKPT)
    print("SW config:", SW_CONFIG)
else:
    print("ENABLE_6STEM=False: BS-Roformer-SWモデルのダウンロードをスキップします。")


## 14. BS-Roformer-SWの元設定を確認


In [ ]:
print(SW_CONFIG.read_text())

## 15. BS-Roformer-SW用の推論設定を構築


In [ ]:
SW_INFERENCE_BATCH_SIZE = 1
SW_INFERENCE_OVERLAP = 4

with SW_CONFIG.open() as f:
    sw_cfg = yaml.load(f, Loader=SafeLoaderWithTuple)

sw_cfg["inference"]["batch_size"] = SW_INFERENCE_BATCH_SIZE
sw_cfg["inference"]["num_overlap"] = SW_INFERENCE_OVERLAP

SW_RUN_CONFIG = SW_MODEL_DIR / "config_colab_improved.yaml"
with SW_RUN_CONFIG.open("w") as f:
    yaml.safe_dump(sw_cfg, f, sort_keys=False)

print("SW batch_size =", SW_INFERENCE_BATCH_SIZE)
print("SW num_overlap =", SW_INFERENCE_OVERLAP)
print("SW runtime config =", SW_RUN_CONFIG)


## 17. BS-Roformer-SWを実行(CUDA)



In [ ]:
if ENABLE_6STEM:
    SW_INPUT_DIR = RUN_ROOT / "input_instrumental_recovered"
    SW_INPUT_DIR.mkdir(exist_ok=True)

    SW_INPUT_WAV = SW_INPUT_DIR / f"{INPUT_WAV.stem}_instrumental_recovered.wav"

    source_for_sw = (
        RECOVERED_WAV
        if ENABLE_VOCAL_AWARE_RECOVERY and RECOVERED_WAV.exists()
        else SOURCE_FOR_EXPORT
    )

    import shutil
    shutil.copy2(source_for_sw, SW_INPUT_WAV)

    print("BS-Roformer-SW input:", SW_INPUT_WAV)
else:
    print("ENABLE_6STEM=False: 6-stem入力準備をスキップします。")


In [ ]:
if ENABLE_6STEM:
    cmd_sw = [
        sys.executable, "-m", "bs_roformer.inference",
        "--model_type", "bs_roformer",
        "--config_path", str(SW_RUN_CONFIG),
        "--model_path", str(SW_CKPT),
        "--input_folder", str(SW_INPUT_DIR),
        "--store_dir", str(OUTPUT_DIR_SW),
        "--device", "cuda:0",
    ]

    print(" ".join(cmd_sw))
    subprocess.run(cmd_sw, check=True)
else:
    print("ENABLE_6STEM=False: BS-Roformer-SW推論をスキップします。")


## 17. BS-Roformer-SWの出力を6-stemに仕分け



In [ ]:
if ENABLE_6STEM:
    sw_raw_wavs = sorted(OUTPUT_DIR_SW.glob("*.wav"))
    print("BS-Roformer-SW raw outputs:")
    for p in sw_raw_wavs:
        print(" ", p.name)

    if not sw_raw_wavs:
        raise RuntimeError("No output WAV files were produced by BS-Roformer-SW.")

    STEM_KEYWORDS = {
        "vocals": ["vocal"],
        "drums": ["drum"],
        "bass": ["bass"],
        "guitar": ["guitar"],
        "piano": ["piano"],
        "other": ["other"],
        "instrumental": ["instrumental"],
    }

    sw_stems = {}
    for stem, keywords in STEM_KEYWORDS.items():
        matches = [p for p in sw_raw_wavs if any(k in p.name.lower() for k in keywords)]
        sw_stems[stem] = matches[0] if matches else None
        print(f"{stem}:", matches[0].name if matches else "見つかりませんでした")
else:
    sw_raw_wavs = []
    sw_stems = {}
    print("ENABLE_6STEM=False: 6-stem出力確認をスキップします。")


## 18. BS-Roformer-SW出力にピークガードを適用してエクスポート


In [ ]:
if ENABLE_6STEM:
    SW_EXPORT_DIR = PROCESSED_DIR / "sw_stems"
    SW_EXPORT_DIR.mkdir(exist_ok=True)

    TARGET_PEAK_DBFS_SW = -1.0

    def apply_peak_guard(path, target_dbfs):
        audio, sr = sf.read(path, always_2d=True, dtype="float32")
        peak = float(np.max(np.abs(audio)))
        target = 10 ** (target_dbfs / 20.0)
        if peak > target and peak > 0:
            audio = audio * (target / peak)
        return audio, sr

    sw_export_paths = {}
    for stem, path in sw_stems.items():
        if path is None:
            continue
        audio, sr = apply_peak_guard(path, TARGET_PEAK_DBFS_SW)
        out_path = SW_EXPORT_DIR / f"{INPUT_WAV.stem}_{stem}_sw_final.wav"
        sf.write(out_path, audio, sr, subtype="FLOAT")
        sw_export_paths[stem] = out_path
        print(f"{stem} ->", out_path)
else:
    sw_export_paths = {}
    print("ENABLE_6STEM=False: 6-stemエクスポートをスキップします。")


## 20. 両分岐の結果を表示


In [ ]:
direct_instrumentals = [
    p for p in raw_wavs
    if any(k in p.name.lower() for k in ["instrumental", "other"])
]

print("=== Branch A: BS-Roformer Revive V2 (2-stem) ===")
print("Vocals:", VOCALS_WAV)
print("Direct model instrumental candidates:")
for p in direct_instrumentals:
    print(" ", p)
print("Residual instrumental (aligned):", RESIDUAL_WAV)
print("Final processed instrumental:", EXPORT_WAV)

print()
print("=== Branch B: BS-Roformer-SW (6-stem) ===")
for stem, path in sw_export_paths.items():
    print(f"{stem}:", path)


- `..._instrumental_residual.wav` — `M - V` で作成した補正前Residual
- `..._instrumental_vocal_recovered.wav` — Vocal-aware Level Recovery後
- `..._instrumental_final.wav` — Peak Guard後の最終Instrumental
- `..._instrumental_recovered` を入力した6-stem出力


## 21. ファイルのダウンロード

添付Notebookと同じく、生成したWAVを `files.download()` で個別に直接ダウンロードします。
ZIP化は行いません。


In [ ]:
from google.colab import files

download_list = [
    VOCALS_WAV,
    RESIDUAL_WAV,
    RECOVERED_WAV,
    EXPORT_WAV,
] + direct_instrumentals + list(sw_export_paths.values())

seen = set()

for p in download_list:
    if p is None:
        continue

    p = Path(p)
    key = str(p.resolve())

    if not p.exists() or key in seen:
        continue

    seen.add(key)

    size_mb = p.stat().st_size / 1024 / 1024
    print(f"Downloading: {p.name} ({size_mb:.2f} MB)")
    files.download(str(p))

print()
print("現在のrunのWAVを個別にダウンロードしました。")
print("別の音声を処理する場合は、アップロードセル（Cell 12）から順番に再実行してください。")
